<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). تمت الترجمة والتحرير بواسطة [كريستينا بوتسكو](https://www.linkedin.com/in/christinabutsko/)، و[نرسيس باجيان](https://www.linkedin.com/in/nersesbagiyan/)، و[يوليا كليموشينا](https://www.linkedin.com/in/yuliya-klimushina-7168a9139)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> الموضوع 4. التصنيف الخطي والانحدار
## <center> الجزء الثالث. مثال توضيحي لتنظيم الانحدار اللوجستي



في المقالة الأولى، أوضحنا كيف تسمح السمات متعددة الحدود للنماذج الخطية ببناء أسطح فاصلة غير خطية. دعونا الآن نظهر هذا بصريا.
دعونا نرى كيف يؤثر التنظيم على جودة التصنيف في مجموعة بيانات حول اختبار الرقائق الدقيقة من دورة Andrew Ng حول التعلم الآلي. سوف نستخدم الانحدار اللوجستي مع ميزات متعددة الحدود ونغير معلمة التنظيم $C$. أولاً، سنرى كيف يؤثر التنظيم على الحدود الفاصلة للمصنف ونتعرف بشكل بديهي على النقص والتجاوز. بعد ذلك، سنختار معلمة التنظيم لتكون قريبة عدديًا من القيمة المثلى عبر (`cross-validation`) و (`GridSearch`).


In [ ]:
# we don't like warnings
# you can comment the following 2 lines if you'd like to
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score)
from sklearn.preprocessing import PolynomialFeatures

لنقم بتحميل البيانات باستخدام `read_csv` من مكتبة `pandas`. في مجموعة البيانات هذه التي تحتوي على 118 شريحة (كائنات)، توجد نتائج لاختبارين لمراقبة الجودة (متغيرين عدديين) ومعلومات عما إذا كانت الرقاقة الدقيقة قد دخلت حيز الإنتاج. تم توسيط المتغيرات بالفعل، مما يعني أن قيم الأعمدة تم طرح قيمها المتوسطة الخاصة بها. وبالتالي، فإن الشريحة الدقيقة "المتوسطة" تتوافق مع قيمة صفر في نتائج الاختبار.  


In [ ]:
# loading data
data = pd.read_csv(
    "../../data/microchip_tests.txt", header=None, names=("test1", "test2", "released")
)
# getting some info about dataframe
data.info()


دعونا نتفحص في الأسطر الخمسة الأولى والأخيرة.


In [ ]:
data.head(5)

In [ ]:
data.tail(5)


يجب علينا الآن حفظ مجموعة التدريب وتسميات الفئة المستهدفة في مصفوفات NumPy منفصلة.


In [ ]:
X = data.iloc[:, :2].values
y = data.iloc[:, 2].values


كخطوة وسيطة، يمكننا رسم البيانات. النقاط البرتقالية تتوافق مع الرقائق المعيبة، والزرقاء تتوافق مع الرقائق العادية.


In [ ]:
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="blue", label="Released")
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="orange", label="Faulty")
plt.xlabel("Test 1")
plt.ylabel("Test 2")
plt.title("2 tests of microchips. Logit with C=1")
plt.legend();


دعونا نحدد دالة لعرض المنحنى الفاصل للمصنف.


In [ ]:
def plot_boundary(clf, X, y, grid_step=0.01, poly_featurizer=None):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, grid_step), np.arange(y_min, y_max, grid_step)
    )

    # to every point from [x_min, m_max]x[y_min, y_max]
    # we put in correspondence its own color
    Z = clf.predict(poly_featurizer.transform(np.c_[xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    plt.contour(xx, yy, Z, cmap=plt.cm.Paired)


نحدد الميزات متعددة الحدود التالية للدرجة $d$ لمتغيرين $x_1$ و$x_2$:
$$\large \{x_1^d, x_1^{d-1}x_2, \ldots x_2^d\} =  \{x_1^ix_2^j\}_{i+j=d, i,j \in \mathbb{N}}$$
على سبيل المثال، بالنسبة إلى $d=3$، ستكون هذه الميزات التالية:
$$\large 1, x_1, x_2,  x_1^2, x_1x_2, x_2^2, x_1^3, x_1^2x_2, x_1x_2^2, x_2^3$$
رسم مثلث فيثاغورس سيوضح عدد هذه الميزات التي ستكون موجودة لـ $d=4,5...$ وما إلى ذلك.
عدد هذه الميزات كبير بشكل كبير، وقد يكون من المكلف إنشاء ميزات متعددة الحدود ذات درجة كبيرة (على سبيل المثال $d=10$) لـ 100 متغير. والأهم من ذلك أنه ليس هناك حاجة إليه. 



سوف نستخدم تطبيق `sklearn` للانحدار اللوجستي. لذلك، قمنا بإنشاء كائن يضيف ميزات متعددة الحدود حتى الدرجة 7 إلى المصفوفة $X$.


In [ ]:
poly = PolynomialFeatures(degree=7)
X_poly = poly.fit_transform(X)

In [ ]:
X_poly.shape


دعونا ندرب الانحدار اللوجستي باستخدام معلمة التنظيم $C = 10^{-2}$.


In [ ]:
C = 1e-2
logit = LogisticRegression(C=C, random_state=17)
logit.fit(X_poly, y)

plot_boundary(logit, X, y, grid_step=0.01, poly_featurizer=poly)

plt.scatter(X[y == 1, 0], X[y == 1, 1], c="blue", label="Released")
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="orange", label="Faulty")
plt.xlabel("Test 1")
plt.ylabel("Test 2")
plt.title("2 tests of microchips. Logit with C=%s" % C)
plt.legend()

print("Accuracy on training set:", round(logit.score(X_poly, y), 3))

يمكننا الآن محاولة زيادة $C$ إلى 1. وبذلك، فإننا نضعف التنظيم، ويمكن أن يحتوي الحل الآن على قيم أكبر (بالقيمة المطلقة) لأوزان النموذج من السابق. الآن تم تحسين دقة المصنف في مجموعة التدريب إلى 0.831.


In [ ]:
C = 1
logit = LogisticRegression(C=C, random_state=17)
logit.fit(X_poly, y)

plot_boundary(logit, X, y, grid_step=0.005, poly_featurizer=poly)

plt.scatter(X[y == 1, 0], X[y == 1, 1], c="blue", label="Released")
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="orange", label="Faulty")
plt.xlabel("Test 1")
plt.ylabel("Test 2")
plt.title("2 tests of microchips. Logit with C=%s" % C)
plt.legend()

print("Accuracy on training set:", round(logit.score(X_poly, y), 3))


إذًا، لماذا لا نزيد $C$ أكثر - حتى 10000؟ الآن، من الواضح أن التنظيم ليس قويًا بما يكفي، ونرى فرطًا في التجهيز. لاحظ أنه مع $C$=1 والحدود "السلسة"، فإن حصة الإجابات الصحيحة في مجموعة التدريب ليست أقل بكثير مما هي عليه هنا. ولكن يمكن للمرء أن يتخيل بسهولة كيف سيعمل نموذجنا الثاني بشكل أفضل بكثير على البيانات الجديدة.


In [ ]:
C = 1e4
logit = LogisticRegression(C=C, random_state=17)
logit.fit(X_poly, y)

plot_boundary(logit, X, y, grid_step=0.005, poly_featurizer=poly)

plt.scatter(X[y == 1, 0], X[y == 1, 1], c="blue", label="Released")
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="orange", label="Faulty")
plt.xlabel("Test 1")
plt.ylabel("Test 2")
plt.title("2 tests of microchips. Logit with C=%s" % C)
plt.legend()

print("Accuracy on training set:", round(logit.score(X_poly, y), 3))


لمناقشة النتائج، دعونا نعيد كتابة الدالة التي تم تحسينها في الانحدار اللوجستي بالنموذج:
$$\large J(X,y,w) = \mathcal{L} + \frac{1}{C}||w||^2,$$
أين
- $\mathcal{L}$ هي دالة الخسارة اللوجستية المجمعة على مجموعة البيانات بأكملها
- $C$ هو معامل التنظيم العكسي (نفس $C$ من تنفيذ `sklearn` لـ `LogisticRegression`)


** المجاميع الفرعية **:
- كلما كانت المعلمة $C$ أكبر، زادت تعقيد العلاقات في البيانات التي يمكن للنموذج استعادتها (يتوافق $C$ بشكل حدسي مع "تعقيد" النموذج - سعة النموذج)
- إذا كان التنظيم قويًا جدًا، أي أن قيم $C$ صغيرة، فقد يكون حل مشكلة تقليل دالة الخسارة اللوجستية هو الحل الذي تكون فيه العديد من الأوزان صغيرة جدًا أو صفرية. النموذج أيضًا لم يتم "معاقبته" بشكل كافٍ بسبب الأخطاء (أي في الدالة $J$، مجموع مربعات الأوزان "يتفوق"، والخطأ $\mathcal{L}$ يمكن أن يكون كبيرًا نسبيًا). في هذه الحالة، سيكون النموذج غير مناسب كما رأينا في حالتنا الأولى.
- على العكس من ذلك، إذا كان التنظيم ضعيفًا جدًا، أي أن قيم $C$ كبيرة، فيمكن أن يصبح المتجه $w$ ذو المكونات ذات القيمة المطلقة العالية هو الحل لمشكلة التحسين. في هذه الحالة، $\mathcal{L}$ لديه مساهمة أكبر في الوظيفة المحسنة $J$. بشكل عام، النموذج "يخشى" جدًا أن يخطئ في تحديد الأشياء من مجموعة التدريب، وبالتالي سوف يفرط في التناسب كما رأينا في الحالة الثالثة.
- الانحدار اللوجستي لن "يفهم" (أو "يتعلم") قيمة $C$ للاختيار كما هو الحال مع الأوزان $w$. وهذا يعني أنه لا يمكن تحديده من خلال حل مشكلة التحسين في الانحدار اللوجستي. لقد رأينا موقفًا مشابهًا من قبل - لا يمكن لشجرة القرار "معرفة" الحد الأقصى للعمق الذي يجب اختياره أثناء عملية التدريب. لذلك، $C$ هو معلمة تشعبية للنموذج تم ضبطها عند التحقق المتبادل؛ وكذلك الحد الأقصى للعمق في الشجرة.



** ضبط معلمة التنظيم **


باستخدام هذا المثال، دعونا نحدد القيمة المثلى لمعلمة التنظيم $C$. يمكن القيام بذلك باستخدام `LogisticRegressionCV` - بحث شبكي للمعلمات متبوعًا بالتحقق المتبادل. تم تصميم هذه الفئة خصيصًا للانحدار اللوجستي (خوارزميات فعالة ذات معلمات بحث معروفة). بالنسبة للنموذج العشوائي، استخدم `GridSearchCV`، `RandomizedSearchCV`، أو خوارزميات خاصة لتحسين المعلمة الفائقة مثل تلك التي تم تنفيذها في `hyperopt`.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

c_values = np.logspace(-2, 3, 500)

logit_searcher = LogisticRegressionCV(Cs=c_values, cv=skf, verbose=1, n_jobs=-1)
logit_searcher.fit(X_poly, y)

In [ ]:
logit_searcher.C_


لمعرفة كيف تختلف جودة النموذج (النسبة المئوية للإجابات الصحيحة في مجموعات التدريب والتحقق من الصحة) مع المعلمة الفائقة $C$، يمكننا رسم الرسم البياني. 


In [ ]:
plt.plot(c_values, np.mean(logit_searcher.scores_[1], axis=0))
plt.xlabel("C")
plt.ylabel("Mean CV-accuracy");


أخيرًا، حدد المنطقة ذات القيم "الأفضل" $C$.


In [ ]:
plt.plot(c_values, np.mean(logit_searcher.scores_[1], axis=0))
plt.xlabel("C")
plt.ylabel("Mean CV-accuracy")
plt.xlim((0, 10));


تذكر أن هذه المنحنيات تسمى منحنيات التحقق. في السابق، قمنا ببنائها يدويًا، ولكن لدى sklearn طرقًا خاصة لإنشاءها والتي سنستخدمها في المستقبل.


### موارد مفيدة
- الطبق الرئيسي [الموقع](https://mlcourse.ai)، [مستودع الدورة](https://github.com/Yorko/mlcourse.ai)، ويوتيوب [القناة](https://www.youtube.com/watch?v=QKTuw4PNOsU&list=PLVlY_7IJCMJeRfZ68eVfEcu-UcN9BbwiX)
- متوسط ["قصة"](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-4-linear-classification-and-regression-44a41b9b5220) استنادًا إلى دفتر الملاحظات هذا
- مواد الدورة التدريبية باعتبارها [مجموعة بيانات Kaggle](https://www.kaggle.com/kashnitsky/mlcourse)
- إذا كنت تقرأ اللغة الروسية: [مقالة](https://habrahabr.ru/company/ods/blog/323890/) على حبراهابر مع ~ نفس المادة. و[محاضرة](https://youtu.be/oTXGQ-_oqvI) على اليوتيوب
- نظرة عامة لطيفة وموجزة على النماذج الخطية مقدمة في كتاب ["التعلم العميق"](http://www.deeplearningbook.org) (I. Goodfellow، Y. Bengio، و A. Courville).
- تتم تغطية النماذج الخطية عمليا في كل كتاب تعلم الآلة. نوصي بـ "التعرف على الأنماط والتعلم الآلي" (C. Bishop) و"التعلم الآلي: منظور احتمالي" (K. Murphy).
- إذا كنت تفضل نظرة شاملة على النموذج الخطي من وجهة نظر الإحصائي، فاطلع على "عناصر التعلم الإحصائي" (T. Hastie، R. Tibshirani، و J. Friedman).
- سيرشدك كتاب "التعلم الآلي أثناء العمل" (P. Harrington) عبر تطبيقات خوارزميات تعلم الآلة الكلاسيكية في لغة بايثون النقية.
- مكتبة [Scikit-learn](http://scikit-learn.org/stable/documentation.html). هؤلاء الرجال يعملون بجد لكتابة وثائق واضحة حقًا.
- Scipy 2017 [برنامج تعليمي لـ scikit-learn](https://github.com/amueller/scipy-2017-sklearn) بواسطة Alex Gramfort وAndreas Mueller.
- [دورة تعلم الآلة] (https://github.com/diefimov/MTH594_MachineLearning) إضافية بمواد جيدة جدًا.
- [تطبيقات](https://github.com/rushter/MLAlgorithms) للعديد من خوارزميات تعلم الآلة. البحث عن الانحدار الخطي والانحدار اللوجستي.